# Option A Pipeline: Detect → Translate

**Goal:** Extract jargon spans from a sentence, then return a plain-language translation for each span.

```
Sentence → [BERT token classifier] jargon spans → [Glossary lookup] translations
```

This notebook does **not** modify `classification_model_version_01.ipynb` or `translate_model_main.py`.

## Checkpoints (stay on track)
| ID | Gate | Pass if |
|---|---|---|
| **CP0** | Scope | Output is `{jargon, translation}` pairs (not sentence rewrite) |
| **CP1** | Span labels | ≥99% of positive rows get a span in the sentence |
| **CP2** | Glossary | Known jargon returns the correct definition |
| **CP3** | Detector scaffold | Token-classifier dataset + model load without error |
| **CP4** | End-to-end | Sample sentences return span + translation pairs |

## CP0 — Imports and data path

**Checkup:** Run the next cell. Confirm `DATA_PATH` exists and row count prints (~6474).

In [3]:
import os
import re
from pathlib import Path

import pandas as pd
import numpy as np
import torch
from torch import nn
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForTokenClassification, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split

DATA_PATH = Path('../Data/final_jargon_dataset.csv')
SPAN_OUT_PATH = Path('../Data/span_labeled_jargon.csv')
GLOSSARY_OUT_PATH = Path('../Data/jargon_glossary.csv')

assert DATA_PATH.exists(), f'Missing dataset: {DATA_PATH.resolve()}'
df = pd.read_csv(DATA_PATH)
print(f'CP0 PASS — loaded {len(df)} rows from {DATA_PATH.resolve()}')
print('columns:', list(df.columns))
print(df['discipline'].value_counts())

CP0 PASS — loaded 6474 rows from /home/hice1/pkim332/VIP_NLP/Data/final_jargon_dataset.csv
columns: ['jargon', 'translation', 'discipline', 'sentence']
discipline
Architecture        1650
Computer Science    1626
No Discipline       1600
Psychology          1598
Name: count, dtype: int64


## CP1 — Build BIO span labels from `(sentence, jargon)`

**Checkup:** After the next cells, `match_rate_positive` should be **≥ 0.99**. If below, stop and fix span matching before training BERT.

In [4]:
LABEL2ID = {'O': 0, 'B-JARGON': 1, 'I-JARGON': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}


def find_jargon_span(sentence, jargon):
    """Return (start, end) character span of jargon in sentence, or None."""
    s = str(sentence).strip().strip('"')
    j = str(jargon).strip()
    if not j or not s:
        return None

    m = re.search(re.escape(j), s, flags=re.IGNORECASE)
    if m:
        return m.start(), m.end()

    # hyphen / whitespace variants (e.g. load-bearing vs load bearing)
    pattern = re.sub(r'[\s\-]+', r'[\\s\\-]+', re.escape(j))
    m = re.search(pattern, s, flags=re.IGNORECASE)
    if m:
        return m.start(), m.end()
    return None


def words_with_char_spans(text):
    """Simple whitespace tokenization with character offsets."""
    words, spans = [], []
    for m in re.finditer(r'\S+', text):
        words.append(m.group())
        spans.append((m.start(), m.end()))
    return words, spans


def bio_tags_for_row(sentence, jargon, discipline):
    s = str(sentence).strip().strip('"')
    words, word_spans = words_with_char_spans(s)
    tags = ['O'] * len(words)

    if discipline == 'No Discipline':
        return s, words, tags, None, 'negative'

    span = find_jargon_span(s, jargon)
    if span is None:
        return s, words, tags, None, 'unmatched'

    start, end = span
    first = True
    for i, (ws, we) in enumerate(word_spans):
        # word overlaps jargon character span
        if we > start and ws < end:
            tags[i] = 'B-JARGON' if first else 'I-JARGON'
            first = False
    return s, words, tags, (start, end), 'matched'

In [5]:
records = []
status_counts = {'matched': 0, 'unmatched': 0, 'negative': 0}

for _, row in df.iterrows():
    sentence_clean, words, tags, span, status = bio_tags_for_row(
        row['sentence'], row['jargon'], row['discipline']
    )
    status_counts[status] += 1
    records.append({
        'jargon': row['jargon'],
        'translation': row['translation'],
        'discipline': row['discipline'],
        'sentence': sentence_clean,
        'words': words,
        'bio_tags': tags,
        'span_start': None if span is None else span[0],
        'span_end': None if span is None else span[1],
        'span_status': status,
    })

span_df = pd.DataFrame(records)
positive = span_df[span_df['discipline'] != 'No Discipline']
match_rate_positive = (positive['span_status'] == 'matched').mean()

print('status_counts:', status_counts)
print(f'match_rate_positive: {match_rate_positive:.4f} ({(positive["span_status"]=="matched").sum()}/{len(positive)})')
print('unmatched examples:')
print(positive[positive['span_status'] == 'unmatched'][['jargon', 'sentence']].head(5).to_string(index=False))

# Persist a readable CSV (space-joined words/tags); in-memory span_df keeps real lists for training.
span_df.assign(
    words=span_df['words'].apply(lambda x: ' '.join(x)),
    bio_tags=span_df['bio_tags'].apply(lambda x: ' '.join(x)),
).to_csv(SPAN_OUT_PATH, index=False)
print(f'saved: {SPAN_OUT_PATH}')

if match_rate_positive >= 0.99:
    print('CP1 PASS')
else:
    print('CP1 FAIL — fix span matching before continuing')

status_counts: {'matched': 4851, 'unmatched': 23, 'negative': 1600}
match_rate_positive: 0.9953 (4851/4874)
unmatched examples:
            jargon                                                                                                                                                                                                                                                                                                    sentence
 Load Bearing Wall                       In architectural design, the identification and preservation of load-bearing walls are crucial to maintaining the structural integrity of a building during renovation or remodeling, as these walls support vertical loads and contribute to the overall stability of the framework.
 Id, Ego, Superego         In Freudian psychology, the interplay between the Id, Ego, and Superego is crucial for understanding the dynamics of human behavior, as the Id represents primal desires, the Ego mediates these urges with rea

## CP2 — Glossary translator (span → definition)

Stage-2 of Option A for known terms: exact/normalized lookup into `jargon → translation`.

**Checkup:** Lookup hit-rate on matched positive rows should be **100%** for exact jargon keys.

In [6]:
class JargonGlossary:
    """Map jargon strings to plain-language translations."""

    def __init__(self, dataframe):
        gloss = (
            dataframe[dataframe['translation'] != 'No translation']
            [['jargon', 'translation', 'discipline']]
            .dropna()
            .drop_duplicates(subset=['jargon'], keep='first')
        )
        self.by_exact = {
            str(r.jargon).strip(): {
                'translation': str(r.translation).strip(),
                'discipline': str(r.discipline).strip(),
            }
            for r in gloss.itertuples(index=False)
        }
        self.by_norm = {self.normalize(k): v for k, v in self.by_exact.items()}

    @staticmethod
    def normalize(text):
        return re.sub(r'[^a-z0-9]+', '', str(text).lower())

    def lookup(self, jargon):
        key = str(jargon).strip()
        if key in self.by_exact:
            return self.by_exact[key]
        norm = self.normalize(key)
        return self.by_norm.get(norm)


glossary = JargonGlossary(df)
glossary_rows = pd.DataFrame([
    {'jargon': k, 'translation': v['translation'], 'discipline': v['discipline']}
    for k, v in glossary.by_exact.items()
])
glossary_rows.to_csv(GLOSSARY_OUT_PATH, index=False)
print(f'glossary size: {len(glossary.by_exact)} → saved {GLOSSARY_OUT_PATH}')

matched_pos = span_df[span_df['span_status'] == 'matched']
hits = matched_pos['jargon'].apply(lambda j: glossary.lookup(j) is not None)
hit_rate = hits.mean()
print(f'glossary hit-rate on matched positives: {hit_rate:.4f}')

sample_j = matched_pos.iloc[0]['jargon']
print('sample lookup:', sample_j, '→', glossary.lookup(sample_j))

if hit_rate == 1.0:
    print('CP2 PASS')
else:
    print('CP2 FAIL — some matched jargon missing from glossary')

glossary size: 4874 → saved ../Data/jargon_glossary.csv
glossary hit-rate on matched positives: 1.0000
sample lookup: Cantilever → {'translation': 'A long projecting beam or girder fixed at only one end, used in bridge construction.', 'discipline': 'Architecture'}
CP2 PASS


## CP3 — BERT token-classifier scaffold (span detector)

Align word-level BIO tags to BERT subword tokens. Train later; this checkpoint only verifies the dataset + model load.

**Checkup:** One batch shapes print correctly and `AutoModelForTokenClassification` loads with 3 labels.

In [7]:
class JargonSpanDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        words = row['words'] if isinstance(row['words'], list) else str(row['words']).split()
        tags = row['bio_tags'] if isinstance(row['bio_tags'], list) else str(row['bio_tags']).split()

        encoding = self.tokenizer(
            words,
            is_split_into_words=True,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt',
        )

        word_ids = encoding.word_ids(batch_index=0)
        label_ids = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != prev_word_id:
                label_ids.append(LABEL2ID[tags[word_id]])
            else:
                # continuation subword: ignore in loss
                label_ids.append(-100)
            prev_word_id = word_id

        return (
            encoding['input_ids'].squeeze(0),
            encoding['attention_mask'].squeeze(0),
            torch.tensor(label_ids, dtype=torch.long),
        )


def prepare_detector_splits(span_dataframe, test_size=0.2, random_state=42):
    usable = span_dataframe[span_dataframe['span_status'] != 'unmatched'].copy()
    train_df, temp_df = train_test_split(usable, test_size=test_size, random_state=random_state)
    valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=random_state)
    return train_df, valid_df, test_df

In [9]:
MODEL_NAME = 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_df, valid_df, test_df = prepare_detector_splits(span_df)
print(f'splits — train: {len(train_df)}, valid: {len(valid_df)}, test: {len(test_df)}')

train_ds = JargonSpanDataset(train_df, tokenizer)
valid_ds = JargonSpanDataset(valid_df, tokenizer)
test_ds = JargonSpanDataset(test_df, tokenizer)

sample_ids, sample_mask, sample_labels = train_ds[0]
print('batch shapes:', tuple(sample_ids.shape), tuple(sample_mask.shape), tuple(sample_labels.shape))
print('label ids present:', sorted(set(sample_labels.tolist())))

detector = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
print(f'model labels: {detector.config.num_labels}')
print('CP3 PASS — dataset + token-classifier ready (training is optional next step)')

splits — train: 5160, valid: 645, test: 646
batch shapes: (128,) (128,) (128,)
label ids present: [-100, 0, 1]


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly

model labels: 3
CP3 PASS — dataset + token-classifier ready (training is optional next step)


## CP4 — End-to-end Option A (detect spans → glossary translate)

Until the detector is trained, this uses **gold spans from the dataset** (or exact jargon match) so you can verify the full I/O contract.

**Checkup:** Each sample prints a list of `{jargon, translation, discipline}` pairs.

In [10]:
def extract_spans_from_bio(words, tags):
    spans = []
    i = 0
    while i < len(words):
        if tags[i] == 'B-JARGON':
            pieces = [words[i]]
            j = i + 1
            while j < len(words) and tags[j] == 'I-JARGON':
                pieces.append(words[j])
                j += 1
            spans.append(' '.join(pieces))
            i = j
        else:
            i += 1
    return spans


def translate_spans(spans, glossary):
    results = []
    for span in spans:
        hit = glossary.lookup(span)
        if hit is None:
            # try stripping light punctuation from token edges
            cleaned = span.strip('.,;:()[]"\'')
            hit = glossary.lookup(cleaned)
            span = cleaned if hit else span
        results.append({
            'jargon': span,
            'translation': None if hit is None else hit['translation'],
            'discipline': None if hit is None else hit['discipline'],
        })
    return results


def option_a_from_gold_row(row, glossary):
    words = row['words'] if isinstance(row['words'], list) else str(row['words']).split()
    tags = row['bio_tags'] if isinstance(row['bio_tags'], list) else str(row['bio_tags']).split()
    spans = extract_spans_from_bio(words, tags)
    return {
        'sentence': row['sentence'],
        'predictions': translate_spans(spans, glossary),
    }


def option_a_pipeline(sentence, glossary, detector_model=None, detector_tokenizer=None, device='cpu'):
    """Full Option A inference.

    If detector_model is None, falls back to glossary key matching inside the sentence
    (useful before the BERT detector is trained).
    """
    s = str(sentence).strip().strip('"')

    if detector_model is None:
        spans = []
        for jargon in sorted(glossary.by_exact.keys(), key=len, reverse=True):
            if find_jargon_span(s, jargon) is not None:
                spans.append(jargon)
        return {'sentence': s, 'predictions': translate_spans(spans, glossary)}

    words, _ = words_with_char_spans(s)
    encoding = detector_tokenizer(
        words,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        max_length=128,
    )
    word_ids = encoding.word_ids(batch_index=0)
    model_inputs = {k: v.to(device) for k, v in encoding.items()}

    detector_model.eval()
    with torch.no_grad():
        logits = detector_model(**model_inputs).logits
    pred_ids = logits.argmax(dim=-1).squeeze(0).tolist()

    tags = ['O'] * len(words)
    for token_idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue
        label = ID2LABEL[pred_ids[token_idx]]
        if tags[word_id] == 'O' and label in ('B-JARGON', 'I-JARGON'):
            tags[word_id] = label

    spans = extract_spans_from_bio(words, tags)
    return {'sentence': s, 'predictions': translate_spans(spans, glossary)}

In [11]:
# Gold-span path (verifies I/O contract with labels you just built)
samples = span_df[span_df['span_status'] == 'matched'].head(3)
for _, row in samples.iterrows():
    out = option_a_from_gold_row(row, glossary)
    print('---')
    print('sentence:', out['sentence'][:120], '...')
    print('predictions:', out['predictions'])

# Glossary-fallback path (works before detector training)
demo_sentence = samples.iloc[0]['sentence']
demo_out = option_a_pipeline(demo_sentence, glossary)
print('---')
print('CP4 glossary-fallback demo:')
print(demo_out)

cp4_ok = all(len(option_a_from_gold_row(r, glossary)['predictions']) >= 1 for _, r in samples.iterrows())
print('CP4 PASS' if cp4_ok else 'CP4 FAIL')

---
sentence: The innovative use of cantilevered structures in modern architecture allows for the seamless extension of building forms ...
predictions: [{'jargon': 'cantilevered', 'translation': 'A structure that projects beyond its support, like a balcony.', 'discipline': 'Architecture'}]
---
sentence: The design of the building's facade not only enhances its aesthetic appeal but also plays a critical role in optimizing  ...
predictions: [{'jargon': 'facade', 'translation': 'The face of a building, especially the principal front that looks onto a street or open space.', 'discipline': 'Architecture'}]
---
sentence: In architectural design, fenestration plays a crucial role in enhancing natural light penetration, optimizing thermal pe ...
predictions: [{'jargon': 'fenestration', 'translation': 'The design and placement of windows in a building.', 'discipline': 'Architecture'}]
---
CP4 glossary-fallback demo:
{'sentence': 'The innovative use of cantilevered structures in modern architect

## Next step (only after CP0–CP4 PASS) — Train the BERT detector

Do **not** start this until CP1–CP4 pass. Training needs GPU time; keep it separate from the pipeline wiring above.

In [12]:
def train_detector(model, train_loader, valid_loader, device, num_epochs=3, lr=2e-5):
    """Minimal training loop for the span detector. Run only after CP0–CP4 pass."""
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(1, num_epochs + 1):
        model.train()
        total_loss = 0.0
        progress = tqdm(train_loader, desc=f'Epoch {epoch}/{num_epochs}')
        for input_ids, attention_mask, labels in progress:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress.set_postfix(loss=loss.item())

        model.eval()
        valid_loss = 0.0
        with torch.no_grad():
            for input_ids, attention_mask, labels in valid_loader:
                input_ids = input_ids.to(device)
                attention_mask = attention_mask.to(device)
                labels = labels.to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                valid_loss += outputs.loss.item()

        print(
            f'epoch {epoch}: train_loss={total_loss/len(train_loader):.4f} '
            f'valid_loss={valid_loss/max(len(valid_loader), 1):.4f}'
        )
    return model


# Uncomment when ready to train:
# device = 'cuda' if torch.cuda.is_available() else 'cpu'
# train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)
# valid_loader = torch.utils.data.DataLoader(valid_ds, batch_size=16)
# detector = train_detector(detector, train_loader, valid_loader, device)
print('Training cell ready — leave commented until CP0–CP4 all PASS.')

Training cell ready — leave commented until CP0–CP4 all PASS.


## Tracking checklist

Run top-to-bottom and mark each gate:

- [ ] **CP0** Dataset loads (~6474 rows)
- [ ] **CP1** `match_rate_positive >= 0.99` and `span_labeled_jargon.csv` written
- [ ] **CP2** Glossary hit-rate = 1.0 and `jargon_glossary.csv` written
- [ ] **CP3** Token dataset batch shapes + BERT token-classifier load
- [ ] **CP4** Sample outputs are lists of `{jargon, translation, discipline}`
- [ ] **Next** Uncomment detector training only after the five checks above